# Regresión logística múltiple: deserción estudiantil

Este cuaderno reproduce el análisis del archivo `modelo_regresion_logistica.py`, paso a paso. Cada celda de código realiza la operación descrita justo antes y guarda los mismos resultados. Ejecute las celdas en orden, desde esta carpeta del proyecto. El cuaderno crea `resultados/` y `visualizaciones/` si hace falta y vuelve a generar sus archivos.

La clase positiva es `desercion = Sí`. El modelo usa `promedio_semestre`, `distancia_km`, `trabaja` y `beca`. Los coeficientes describen asociaciones condicionales; no prueban causalidad.


## Preparación: comprobar dependencias y definir rutas

Primero se buscan las librerías necesarias. Si falta alguna, se instala con el mismo Python que está ejecutando el cuaderno. Después se importan los paquetes y se definen las rutas con relación a la carpeta del proyecto.


In [ ]:
# Comprobar e instalar dependencias antes de importarlas.
import importlib.util
import subprocess
import sys

DEPENDENCIAS = {
    "pandas": "pandas", "numpy": "numpy", "sklearn": "scikit-learn",
    "statsmodels": "statsmodels", "matplotlib": "matplotlib", "openpyxl": "openpyxl",
}
faltan = [paquete for modulo, paquete in DEPENDENCIAS.items()
          if importlib.util.find_spec(modulo) is None]
if faltan:
    print("Instalando:", ", ".join(faltan))
    subprocess.check_call([sys.executable, "-m", "pip", "install", *faltan])
else:
    print("Todas las dependencias están disponibles.")


In [ ]:
# Importar las librerías después de la comprobación y preparar carpetas.
from pathlib import Path
import warnings

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
from IPython.display import display, Image
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             confusion_matrix, roc_auc_score, roc_curve)
from sklearn.model_selection import train_test_split

CARPETA = Path.cwd().resolve()
CARPETA_DATOS = CARPETA / "datos"
CARPETA_RESULTADOS = CARPETA / "resultados"
CARPETA_FIGURAS = CARPETA / "visualizaciones"
CARPETA_RESULTADOS.mkdir(exist_ok=True)
CARPETA_FIGURAS.mkdir(exist_ok=True)
ARCHIVO_DATOS = CARPETA_DATOS / "datos.xlsx"
ARCHIVO_EXCEL = CARPETA_RESULTADOS / "regresion_logistica_resultados.xlsx"
print("Carpeta del proyecto:", CARPETA)


## 1. Cargar y describir los datos

Se lee la hoja `estudiantes`. Revisamos dimensiones y valores faltantes antes de modelar. El porcentaje positivo se calcula como la proporción de filas cuyo resultado es Sí. El detalle se exportará a `descripcion` y `detalle_faltantes`.


In [ ]:
# Leer el archivo y resumir su contenido.
def normalizar_si_no(serie):
    texto = serie.astype("string").str.strip().str.lower()
    texto = texto.str.replace("í", "i", regex=False).str.replace("�", "i", regex=False)
    return texto.map({"si": 1, "no": 0})

datos = pd.read_excel(ARCHIVO_DATOS, sheet_name="estudiantes")
faltantes_por_columna = datos.isna().sum()
total_faltantes = int(faltantes_por_columna.sum())
y_original = normalizar_si_no(datos["desercion"])
if y_original.isna().any():
    raise ValueError("desercion debe contener únicamente Sí y No.")
porcentaje_si = float(y_original.mean() * 100)
print(f"Filas: {len(datos)} | columnas: {datos.shape[1]} | celdas faltantes: {total_faltantes}")
print(f"Deserción Sí: {int(y_original.sum())} ({porcentaje_si:.1f} %)")
display(datos.head())
display(faltantes_por_columna.rename_axis("columna").reset_index(name="datos_faltantes"))
descripcion = pd.DataFrame({
    "indicador": ["filas", "columnas", "celdas_faltantes", "porcentaje_desercion_si", "casos_desercion_si", "casos_desercion_no", "faltantes_por_columna"],
    "valor": [len(datos), datos.shape[1], total_faltantes, porcentaje_si,
              int(y_original.sum()), int((y_original == 0).sum()), "Ver hoja detalle_faltantes"],
})
detalle_faltantes = faltantes_por_columna.rename_axis("columna").reset_index(name="datos_faltantes")


## 2. Convertir variables a números

La regresión logística necesita valores numéricos. Codificamos Sí como 1 y No como 0 para la respuesta y para `trabaja` y `beca`. Así, No es la categoría de referencia para esas dos variables. Las variables continuas conservan su escala original.


In [ ]:
# Preparar exactamente los cuatro predictores solicitados y la respuesta.
PREDICTORES = ["promedio_semestre", "distancia_km", "trabaja", "beca"]
modelado = pd.DataFrame(index=datos.index)
modelado["promedio_semestre"] = pd.to_numeric(datos["promedio_semestre"], errors="coerce")
modelado["distancia_km"] = pd.to_numeric(datos["distancia_km"], errors="coerce")
modelado["trabaja"] = normalizar_si_no(datos["trabaja"])
modelado["beca"] = normalizar_si_no(datos["beca"])
modelado["desercion"] = y_original
if modelado.isna().any().any():
    raise ValueError("Hay faltantes o categorías distintas de Sí/No en las variables del modelo.")
X = modelado[PREDICTORES].astype(float)
y = modelado["desercion"].astype(int)
print("Codificación: desercion Sí=1/No=0; trabaja Sí=1/No=0 (referencia No); beca Sí=1/No=0 (referencia No).")
display(modelado.head())


## 3. Dividir entrenamiento y prueba

El entrenamiento (75 %) se usa para estimar el modelo. La prueba (25 %) queda apartada hasta la evaluación. La estratificación mantiene una proporción parecida de Sí y No en ambos grupos. La semilla 42 hace reproducible la partición.


In [ ]:
# Separar los casos conservando la proporción de clases.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
print(f"Entrenamiento: {len(y_train)} filas; positivos={y_train.mean():.1%}.")
print(f"Prueba: {len(y_test)} filas; positivos={y_test.mean():.1%}.")


## 4. Ajustar la regresión logística y leer los coeficientes

La regresión logística combina los predictores en `z` y convierte ese valor en probabilidad con la función sigmoide `p = 1 / (1 + e^-z)`. scikit-learn se emplea para predecir; statsmodels calcula valores p e intervalos de confianza. El odds ratio `exp(β)` indica por cuánto se multiplican las chances, manteniendo fijas las otras variables; no es un cambio directo en puntos de probabilidad.

Un beta positivo se asocia con mayores chances estimadas y uno negativo con menores chances. Para las variables binarias, la comparación es Sí frente a No. Esta interpretación es asociativa, no causal.


In [ ]:
# Ajustar ambos modelos usando únicamente el conjunto de entrenamiento.
modelo = LogisticRegression(random_state=42, max_iter=1000)
modelo.fit(X_train, y_train)
X_train_sm = sm.add_constant(X_train, has_constant="add")
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    modelo_sm = sm.Logit(y_train, X_train_sm).fit(disp=False, maxiter=200)

parametros = modelo_sm.params
intervalos = modelo_sm.conf_int()
pvalores = modelo_sm.pvalues
nombres = {
    "const": "Intercepto (todas las X=0)",
    "promedio_semestre": "promedio_semestre (por +1 punto)",
    "distancia_km": "distancia_km (por +1 km)",
    "trabaja": "trabaja: Sí frente a No (referencia No)",
    "beca": "beca: Sí frente a No (referencia No)",
}
interpretaciones = []
for nombre, beta in parametros.items():
    if nombre == "const":
        frase = "Log-chances cuando promedio=0, distancia=0, trabaja=No y beca=No; puede no ser un perfil realista."
    else:
        direccion = "mayores" if beta > 0 else "menores" if beta < 0 else "iguales"
        frase = f"Beta {beta:+.3f}: se asocia con chances estimadas {direccion} de deserción, manteniendo las otras variables constantes."
    interpretaciones.append(frase)
coeficientes = pd.DataFrame({
    "variable": [nombres.get(n, n) for n in parametros.index],
    "beta": parametros.values,
    "IC95_inferior_beta": intervalos[0].values,
    "IC95_superior_beta": intervalos[1].values,
    "valor_p": pvalores.values,
    "odds_ratio_exp_beta": np.exp(parametros.values),
    "exp_IC95_inferior": np.exp(intervalos[0].values),
    "exp_IC95_superior": np.exp(intervalos[1].values),
    "interpretacion_del_signo": interpretaciones,
})
ecuacion = "p = 1 / (1 + e^-z), donde z = " + " + ".join(
    f"({parametros[n]:.6f} * {n})" for n in parametros.index if n != "const"
) + f" + ({parametros['const']:.6f})"
print("Ecuación estimada:", ecuacion)
display(coeficientes)


## 5. Graficar la curva sigmoide

En un modelo múltiple la curva depende de varias variables. Para verla en dos dimensiones, variamos `promedio_semestre`, fijamos distancia en su media y dejamos `trabaja=No` y `beca=No`. La pendiente descendente indica que, en este perfil, promedios mayores se asocian con una probabilidad estimada menor. Este gráfico muestra una sección del modelo, no todos los perfiles posibles.


In [ ]:
# Trazar probabilidad estimada al variar el promedio y guardar PNG.
rango_promedio = np.linspace(X["promedio_semestre"].min(), X["promedio_semestre"].max(), 250)
perfil_sigmoide = pd.DataFrame({
    "promedio_semestre": rango_promedio,
    "distancia_km": X["distancia_km"].mean(),
    "trabaja": 0,
    "beca": 0,
})
prob_sigmoide = modelo.predict_proba(perfil_sigmoide[PREDICTORES])[:, 1]
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(rango_promedio, prob_sigmoide, color="#2878B5", lw=2.5)
ax.set(title="Curva sigmoide: probabilidad estimada de deserción",
       xlabel="Promedio del semestre", ylabel="Probabilidad estimada de deserción (Sí)", ylim=(0, 1))
ax.text(0.02, 0.03, "Distancia en su media; trabaja=No y beca=No", transform=ax.transAxes, fontsize=9)
ax.grid(alpha=0.25)
fig.tight_layout()
ruta_sigmoide = CARPETA_FIGURAS / "04_curva_sigmoide.png"
fig.savefig(ruta_sigmoide, dpi=150, bbox_inches="tight")
plt.close(fig)
display(Image(filename=str(ruta_sigmoide)))


## 6. Distribución de probabilidades predichas

El histograma reúne las probabilidades calculadas para los casos de prueba y las colorea según su valor real. Cuanto menos se superpongan las barras de Sí y No, más fácil es separar ambas clases. El histograma por sí solo no elige un umbral ni resume todos los aspectos del desempeño.


In [ ]:
# Predecir las probabilidades de prueba y comparar sus distribuciones.
prob_test = modelo.predict_proba(X_test)[:, 1]
fig, ax = plt.subplots(figsize=(8, 5))
for clase, etiqueta, color in [(0, "No desertó (No)", "#2878B5"), (1, "Desertó (Sí)", "#D9534F")]:
    ax.hist(prob_test[y_test.to_numpy() == clase], bins=20, alpha=0.60,
            label=etiqueta, color=color, edgecolor="white")
ax.set(title="Probabilidad predicha de deserción en prueba",
       xlabel="Probabilidad estimada de deserción (Sí)", ylabel="Número de estudiantes")
ax.legend()
fig.tight_layout()
ruta_prob = CARPETA_FIGURAS / "01_probabilidades.png"
fig.savefig(ruta_prob, dpi=150, bbox_inches="tight")
plt.close(fig)
display(Image(filename=str(ruta_prob)))


## 7. Matriz de confusión con umbral 0,5

Si la probabilidad predicha es al menos 0,5, asignamos la clase Sí. VP son positivos detectados; FN son positivos que se escaparon; FP son alertas falsas; VN son negativos correctamente reconocidos. Para este problema, los FN son especialmente importantes porque representan estudiantes que desertaron y no fueron señalados.


In [ ]:
# Clasificar, contar los cuatro resultados y guardar tabla y figura.
umbral_base = 0.5
pred_test = (prob_test >= umbral_base).astype(int)
mat = confusion_matrix(y_test, pred_test, labels=[1, 0])
vp, fn, fp, vn = map(int, [mat[0, 0], mat[0, 1], mat[1, 0], mat[1, 1]])
matriz_confusion = pd.DataFrame([
    {"celda":"VP", "real":"Sí", "predicho":"Sí", "conteo":vp, "significado":"Desertó y el modelo lo detectó."},
    {"celda":"FN", "real":"Sí", "predicho":"No", "conteo":fn, "significado":"Desertó y el modelo no lo detectó."},
    {"celda":"FP", "real":"No", "predicho":"Sí", "conteo":fp, "significado":"No desertó, pero el modelo alertó."},
    {"celda":"VN", "real":"No", "predicho":"No", "conteo":vn, "significado":"No desertó y el modelo tampoco alertó."},
])
display(matriz_confusion)
fig, ax = plt.subplots(figsize=(6.5, 5.2))
im = ax.imshow([[vp, fn], [fp, vn]], cmap="Blues")
ax.set_xticks([0, 1], labels=["Predicho: Sí", "Predicho: No"])
ax.set_yticks([0, 1], labels=["Real: Sí", "Real: No"])
ax.set_title("Matriz de confusión (umbral = 0,5)")
for (i, j), (sigla, valor) in zip([(0,0),(0,1),(1,0),(1,1)], [("VP",vp),("FN",fn),("FP",fp),("VN",vn)]):
    ax.text(j, i, f"{sigla}\n{valor}", ha="center", va="center", fontsize=14)
fig.colorbar(im, ax=ax, label="Conteo")
fig.tight_layout()
ruta_matriz = CARPETA_FIGURAS / "02_matriz_confusion.png"
fig.savefig(ruta_matriz, dpi=150, bbox_inches="tight")
plt.close(fig)
display(Image(filename=str(ruta_matriz)))
print(f"Con umbral 0,5: VP={vp}, FN={fn}, FP={fp}, VN={vn}.")


## 8. Métricas y comparación con una línea base

Accuracy mide la proporción de aciertos totales; precisión, qué proporción de las alertas Sí fue correcta; sensibilidad, qué proporción de quienes sí desertaron fue detectada; AUC evalúa la capacidad de ordenar positivos por encima de negativos. La línea base siempre predice la clase más frecuente del entrenamiento. Si las métricas de entrenamiento superan mucho a las de prueba puede haber sobreajuste; una diferencia pequeña no lo descarta de forma definitiva.


In [ ]:
# Calcular métricas para train y test y compararlas con la clase mayoritaria.
def metricas_clasificacion(nombre, reales, probabilidades, predicciones):
    return {"conjunto": nombre,
            "accuracy": accuracy_score(reales, predicciones),
            "precision": precision_score(reales, predicciones, zero_division=0),
            "recall_sensibilidad": recall_score(reales, predicciones, zero_division=0),
            "AUC_ROC": roc_auc_score(reales, probabilidades)}

prob_train = modelo.predict_proba(X_train)[:, 1]
pred_train = (prob_train >= umbral_base).astype(int)
clase_mayoritaria = int(y_train.value_counts().idxmax())
metricas = pd.DataFrame([
    metricas_clasificacion("Modelo - entrenamiento", y_train, prob_train, pred_train),
    metricas_clasificacion("Modelo - prueba", y_test, prob_test, pred_test),
    metricas_clasificacion("Línea base - entrenamiento (siempre clase mayoritaria)", y_train,
                           np.full(len(y_train), float(clase_mayoritaria)), np.full(len(y_train), clase_mayoritaria)),
    metricas_clasificacion("Línea base - prueba (siempre clase mayoritaria)", y_test,
                           np.full(len(y_test), float(clase_mayoritaria)), np.full(len(y_test), clase_mayoritaria)),
])
lectura_metricas = pd.DataFrame([
    {"metrica":"accuracy", "lectura_sencilla":"De cada 10 estudiantes, cuántos clasifica correctamente, aproximadamente."},
    {"metrica":"precision", "lectura_sencilla":"De cada 10 alertas de deserción, cuántas corresponden a estudiantes que sí desertan."},
    {"metrica":"recall_sensibilidad", "lectura_sencilla":"De cada 10 estudiantes que sí desertan, a cuántos detecta el modelo."},
    {"metrica":"AUC_ROC", "lectura_sencilla":"Probabilidad de ordenar a un desertor por encima de un no desertor; 0,5 es azar."},
])
display(metricas.round(3))
display(lectura_metricas)
modelo_train = metricas.iloc[0]
modelo_prueba = metricas.iloc[1]
dif_auc = float(modelo_train["AUC_ROC"] - modelo_prueba["AUC_ROC"])
dif_acc = float(modelo_train["accuracy"] - modelo_prueba["accuracy"])
sobreajuste = ("Posible sobreajuste: revisar la brecha entre entrenamiento y prueba."
               if dif_auc > 0.10 or dif_acc > 0.10 else
               "No se observa una brecha grande; no hay señal clara de sobreajuste en esta comparación.")
print(sobreajuste)
print(f"En prueba: de cada 10, acierta aproximadamente {modelo_prueba['accuracy']*10:.1f}; "
      f"de cada 10 alertas, {modelo_prueba['precision']*10:.1f} son correctas; "
      f"detecta {modelo_prueba['recall_sensibilidad']*10:.1f} de cada 10 positivos.")


## 9. Curva ROC y umbrales

La curva ROC representa sensibilidad frente a tasa de falsas alarmas. La diagonal indica el desempeño esperado al azar. AUC más cerca de 1 indica mejor ordenamiento de las clases; 0,5 corresponde al azar. Los puntos muestran los umbrales 0,2, 0,5 y 0,8.

Después comparamos umbrales de 0,1 a 0,9. Sensibilidad alta ayuda a que se escapen menos positivos, pero suele crear más falsas alarmas. El umbral recomendado maximiza sensibilidad en esta tabla y, si hay empate, toma la mayor precisión. La tabla se calcula sobre prueba para ilustrar resultados; el umbral final debe confirmarse con datos de validación.


In [ ]:
# Dibujar ROC con AUC, diagonal aleatoria y puntos de tres umbrales.
fpr, tpr, _ = roc_curve(y_test, prob_test)
auc_prueba = roc_auc_score(y_test, prob_test)
fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(fpr, tpr, color="#2878B5", lw=2, label=f"Modelo (AUC = {auc_prueba:.3f})")
ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Azar")
for t in (0.2, 0.5, 0.8):
    p = (prob_test >= t).astype(int)
    tn_t, fp_t, fn_t, tp_t = confusion_matrix(y_test, p, labels=[0, 1]).ravel()
    sens = tp_t / (tp_t + fn_t) if tp_t + fn_t else 0
    tasa_fp = fp_t / (fp_t + tn_t) if fp_t + tn_t else 0
    ax.scatter(tasa_fp, sens, s=55, label=f"Umbral {t:.1f}")
    ax.annotate(f"{t:.1f}", (tasa_fp, sens), xytext=(5, 5), textcoords="offset points")
ax.set(xlabel="Tasa de falsas alarmas (FPR)", ylabel="Sensibilidad (TPR)",
       title="Curva ROC en prueba", xlim=(0,1), ylim=(0,1))
ax.legend(loc="lower right")
fig.tight_layout()
ruta_roc = CARPETA_FIGURAS / "03_curva_roc.png"
fig.savefig(ruta_roc, dpi=150, bbox_inches="tight")
plt.close(fig)
display(Image(filename=str(ruta_roc)))
print(f"AUC en prueba: {auc_prueba:.3f}")


In [ ]:
# Medir sensibilidad, precisión y falsas alarmas para nueve umbrales.
filas_umbral = []
for t in np.round(np.arange(0.1, 1.0, 0.1), 1):
    p = (prob_test >= t).astype(int)
    tn_t, fp_t, fn_t, tp_t = confusion_matrix(y_test, p, labels=[0, 1]).ravel()
    filas_umbral.append({
        "umbral": t,
        "sensibilidad_recall": tp_t / (tp_t + fn_t) if tp_t + fn_t else 0,
        "precision": tp_t / (tp_t + fp_t) if tp_t + fp_t else 0,
        "falsas_alarmas_FPR": fp_t / (fp_t + tn_t) if fp_t + tn_t else 0,
        "VP": tp_t, "FN": fn_t, "FP": fp_t, "VN": tn_t,
    })
umbrales = pd.DataFrame(filas_umbral)
display(umbrales.round(3))
# Graficar el intercambio entre precisión y sensibilidad.
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(umbrales["umbral"], umbrales["precision"], marker="o", lw=2, label="Precisión")
ax.plot(umbrales["umbral"], umbrales["sensibilidad_recall"], marker="o", lw=2, label="Sensibilidad (recall)")
ax.set(title="Precisión y sensibilidad según el umbral", xlabel="Umbral de clasificación",
       ylabel="Proporción", xticks=umbrales["umbral"], ylim=(0, 1.05))
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()
ruta_precision_recall = CARPETA_FIGURAS / "05_precision_sensibilidad_umbral.png"
fig.savefig(ruta_precision_recall, dpi=150, bbox_inches="tight")
plt.close(fig)
display(Image(filename=str(ruta_precision_recall)))
# Priorizar sensibilidad; desempatar por precisión y luego por mayor umbral.
recomendado = umbrales.sort_values(["sensibilidad_recall", "precision", "umbral"],
                                   ascending=[False, False, False]).iloc[0]
recomendacion = (f"Umbral sugerido: {recomendado['umbral']:.1f}; sensibilidad "
                 f"{recomendado['sensibilidad_recall']:.1%}, precisión {recomendado['precision']:.1%}, "
                 f"falsas alarmas {recomendado['falsas_alarmas_FPR']:.1%} en prueba.")
print(recomendacion)


## 10. Guardar predicciones y el libro de resultados

La tabla de predicciones permite examinar caso por caso el valor real, la probabilidad estimada y la clase asignada con el umbral 0,5. Al final guardamos todas las tablas en un solo Excel, cada una en su propia hoja, y las cifras clave en `resumen`.


In [ ]:
# Preparar predicciones para los casos de prueba.
predicciones = pd.DataFrame({
    "indice_fila_origen": X_test.index,
    "valor_real": np.where(y_test.to_numpy() == 1, "Sí", "No"),
    "probabilidad_desercion_si": prob_test,
    "clase_predicha_umbral_0_5": np.where(pred_test == 1, "Sí", "No"),
}).sort_values("indice_fila_origen")
print(f"Predicciones de prueba: {len(predicciones)} filas.")
display(predicciones.head(10))


In [ ]:
# Reunir cifras clave y exportar tablas con openpyxl a un libro Excel.
resumen = pd.DataFrame([
    {"cifra_clave":"Filas analizadas", "valor":len(datos), "lectura":"Registros en estudiantes."},
    {"cifra_clave":"Porcentaje desercion Sí", "valor":porcentaje_si, "lectura":"Porcentaje de clase positiva."},
    {"cifra_clave":"Accuracy modelo prueba", "valor":float(modelo_prueba["accuracy"]), "lectura":"Aciertos en prueba con umbral 0,5."},
    {"cifra_clave":"Precision modelo prueba", "valor":float(modelo_prueba["precision"]), "lectura":"Proporción de alertas correctas."},
    {"cifra_clave":"Sensibilidad modelo prueba", "valor":float(modelo_prueba["recall_sensibilidad"]), "lectura":"Proporción de positivos detectados."},
    {"cifra_clave":"AUC modelo prueba", "valor":auc_prueba, "lectura":"Capacidad de ordenar positivos sobre negativos."},
    {"cifra_clave":"AUC diferencia train-prueba", "valor":dif_auc, "lectura":"Brecha entre ambas particiones."},
    {"cifra_clave":"Revisión sobreajuste", "valor":sobreajuste, "lectura":"Comparación descriptiva, no prueba definitiva."},
    {"cifra_clave":"Umbral sugerido sensibilidad", "valor":float(recomendado["umbral"]), "lectura":recomendacion},
    {"cifra_clave":"Ecuación logística estimada", "valor":ecuacion, "lectura":"z suma betas por X más intercepto."},
    {"cifra_clave":"VP", "valor":vp, "lectura":"Desertó y fue detectado."},
    {"cifra_clave":"FN", "valor":fn, "lectura":"Desertó y no fue detectado."},
    {"cifra_clave":"FP", "valor":fp, "lectura":"No desertó, pero recibió alerta."},
    {"cifra_clave":"VN", "valor":vn, "lectura":"No desertó y no recibió alerta."},
])
with pd.ExcelWriter(ARCHIVO_EXCEL, engine="openpyxl") as escritor:
    descripcion.to_excel(escritor, sheet_name="descripcion", index=False)
    detalle_faltantes.to_excel(escritor, sheet_name="detalle_faltantes", index=False)
    coeficientes.to_excel(escritor, sheet_name="coeficientes", index=False)
    matriz_confusion.to_excel(escritor, sheet_name="matriz_confusion", index=False)
    metricas.to_excel(escritor, sheet_name="metricas", index=False)
    lectura_metricas.to_excel(escritor, sheet_name="lectura_metricas", index=False)
    umbrales.to_excel(escritor, sheet_name="umbrales", index=False)
    predicciones.to_excel(escritor, sheet_name="predicciones", index=False)
    resumen.to_excel(escritor, sheet_name="resumen", index=False)
print("Excel guardado:", ARCHIVO_EXCEL)
print("Figuras guardadas en:", CARPETA_FIGURAS)
display(resumen)


## Interpretación general y límites

En los datos analizados, 28,5 % de estudiantes desertó. La prueba obtiene accuracy de aproximadamente 7,8 aciertos por cada 10 casos, pero con umbral 0,5 detecta cerca de 3,9 de cada 10 desertores; AUC resume la separación general, sin depender de un umbral concreto. Al umbral recomendado de 0,1 la sensibilidad sube, pero muchas personas que no desertarían también reciben alerta. El equipo debe considerar esa carga y validar el umbral con otros datos.

Las asociaciones estimadas no prueban causalidad. Puede haber variables importantes que no se midieron; el desempeño puede cambiar con otros estudiantes o periodos; una sola partición no cuantifica toda esa variabilidad. No conviene usar la predicción como única base para decisiones sobre estudiantes.
